# Scorer Quality

Leaderboards for the scorer evaluation metrics PyRIT already tracks under
`pyrit/datasets/scorer_evals/`. See [Scorer Metrics](../code/scoring/4_scorer_metrics.ipynb) for
what each metric means and how these numbers are produced.

## Objective Scorer Leaderboard

Objective scorers answer a true/false question (e.g. "was the objective achieved?"). Ranked by
F1 score, the harmonic mean of precision and recall.

In [ ]:
import pandas as pd

from pyrit.score import get_all_objective_metrics
from pyrit.setup import IN_MEMORY, initialize_pyrit_async

await initialize_pyrit_async(memory_db_type=IN_MEMORY, silent=True)  # type: ignore

objective_metrics = get_all_objective_metrics()
objective_metrics.sort(key=lambda entry: entry.metrics.f1_score, reverse=True)

objective_rows = [
    {
        "Name": entry.scorer_identifier.unique_name,
        "Accuracy": entry.metrics.accuracy,
        "F1 Score": entry.metrics.f1_score,
        "Precision": entry.metrics.precision,
        "Recall": entry.metrics.recall,
        "Samples": entry.metrics.num_responses,
    }
    for entry in objective_metrics
]

objective_df = pd.DataFrame(objective_rows)
pd.set_option("display.max_rows", None)
print(objective_df.to_string(index=False))

Auto-discovered plaintext environment file ./.pyrit/.env will be loaded. Azure Key Vault through env_akv_ref is more secure for shared or deployed secrets; use .env.local only for deliberate local overrides. To inspect a resolved AKV-only configuration from a source checkout, run `python -m build_scripts.export_akv_environment`; it writes ~/.pyrit/.env_akv.


                               Name  Accuracy  F1 Score  Precision   Recall  Samples
  TrueFalseInverterScorer::e7af90c2  0.896203  0.893506   0.886598 0.900524      395
  TrueFalseInverterScorer::9e875a98  0.893671  0.891753   0.878173 0.905759      395
  TrueFalseInverterScorer::9355fe9c  0.878481  0.875000   0.870466 0.879581      395
  TrueFalseInverterScorer::f31c9af8  0.883544  0.875000   0.909605 0.842932      395
  TrueFalseInverterScorer::7383238a  0.848404  0.860636   0.792793 0.941176      376
  TrueFalseInverterScorer::4c10ed71  0.845570  0.853012   0.790179 0.926702      395
  TrueFalseInverterScorer::cc1b3ff2  0.792553  0.825893   0.708812 0.989305      376
  TrueFalseInverterScorer::4c6b1acf  0.784574  0.820399   0.700758 0.989305      376
   SelfAskTrueFalseScorer::b0079ec4  0.800000  0.769679   0.868421 0.691099      395
   SelfAskTrueFalseScorer::f4f59053  0.792405  0.755952   0.875862 0.664921      395
   SelfAskTrueFalseScorer::64fbaf40  0.756962  0.733333   0.78106

## Harm Scorer Leaderboard

Harm scorers produce a severity score (0.0-1.0). Ranked by `krippendorff_alpha_combined` —
agreement between the model's scores and human raters, ranging from -1.0 (systematic
disagreement) to 1.0 (perfect agreement) — across every harm category PyRIT currently has
metrics for. Alpha isn't comparable *across* categories (each has its own human-labeled
dataset), so treat this as one leaderboard per category, stacked into a single table for
convenience.

In [ ]:
from pyrit.common.path import SCORER_EVALS_HARM_PATH
from pyrit.score import get_all_harm_metrics

# Harm categories are discovered from the files present on disk rather than a hardcoded list,
# so a newly added category shows up here without a code change.
harm_categories = sorted(
    path.name.removesuffix("_metrics.jsonl") for path in SCORER_EVALS_HARM_PATH.glob("*_metrics.jsonl")
)

harm_metrics = [
    (harm_category, entry)
    for harm_category in harm_categories
    for entry in get_all_harm_metrics(harm_category=harm_category)
]
harm_metrics.sort(key=lambda item: item[1].metrics.krippendorff_alpha_combined, reverse=True)

harm_rows = [
    {
        "Name": entry.scorer_identifier.unique_name,
        "Harm Category": harm_category,
        "MAE": entry.metrics.mean_absolute_error,
        "Alpha Combined": entry.metrics.krippendorff_alpha_combined,
        "Alpha Humans": entry.metrics.krippendorff_alpha_humans,
        "Alpha Model": entry.metrics.krippendorff_alpha_model,
        "Samples": entry.metrics.num_responses,
    }
    for harm_category, entry in harm_metrics
]

harm_df = pd.DataFrame(harm_rows)
print(harm_df.to_string(index=False))

                              Name         Harm Category      MAE  Alpha Combined Alpha Humans  Alpha Model  Samples
     SelfAskLikertScorer::ce5da81b                sexual 0.131410        0.901870         None     0.981653       78
AzureContentFilterScorer::1a9b9789           hate_speech 0.171308        0.861625         None     1.000000       59
     SelfAskLikertScorer::ce31ba14           hate_speech 0.169492        0.853867         None     0.951764       59
     SelfAskLikertScorer::772e51c8              violence 0.161458        0.850706         None     0.933544       96
     SelfAskLikertScorer::1607ea13           hate_speech 0.173729        0.850186         None     0.948099       59
     SelfAskLikertScorer::8e9637f6           hate_speech 0.177966        0.849843         None     0.950146       59
     SelfAskLikertScorer::118336dd              violence 0.174479        0.845945         None     0.959117       96
     SelfAskLikertScorer::67df3fb9              violence 0.17708

## Note on scope

`get_all_objective_metrics()` reads `objective/objective_achieved_metrics.jsonl` only, matching
how it's documented and used elsewhere in PyRIT. A separate `refusal_scorer/refusal_metrics.jsonl`
registry evaluates refusal scorers against its own human-labeled dataset, using the same
`ObjectiveScorerMetrics` shape. It isn't merged into the leaderboard above because it measures a
different task (refusal detection, not objective achievement) against a different ground truth
set, and mixing the two would make the F1 ranking misleading. A follow-up could add it as its own
leaderboard.